In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from scipy.stats import binomtest, mannwhitneyu, fisher_exact, kruskal
from IPython.display import display, HTML, Markdown

# ── Database connection ──
DB_PATH = "C:/Users/scgee/OneDrive/Documents/Projects/PatientPunk/polina_onemonth.db"
conn = sqlite3.connect(DB_PATH)

# ── Sentiment mapping ──
SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

def to_numeric(s):
    """Convert sentiment string to numeric score."""
    return SENTIMENT_SCORE.get(s, 0.0)

def classify_outcome(avg_score):
    """Classify user-level average into outcome category."""
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    return "mixed/neutral"

def wilson_ci(k, n, z=1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0, center - margin), min(1, center + margin)

def nnt(treatment_rate, baseline_rate):
    """Number needed to treat. Returns None if rates are equal or inverted."""
    diff = treatment_rate - baseline_rate
    if diff <= 0:
        return None
    return round(1 / diff, 1)

# ── Chart defaults ──
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# ── Filtering sets ──
GENERIC_TERMS = {
    "supplements", "medication", "treatment", "therapy", "drug", "drugs",
    "vitamin", "prescription", "pill", "pills", "dosage", "dose",
}

# Colors
COLORS = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}


**Research Question:** "Which treatments have the best outcomes in Long COVID?"

## Abstract

Among 1,121 users reporting on treatments in r/covidlonghaulers (March 11 -- April 10, 2026), we identified 94 treatments with at least 10 user-level reports after excluding generic terms and causal-context vaccines. The community's top-performing treatments span several categories: **micronutrients** (magnesium at 93% positive, quercetin at 96%, electrolytes at 88%), **immune modulators** (low dose naltrexone at 74% positive across 183 users --- the largest sample), and **novel interventions** (nicotine patches at 71% positive, GLP-1 receptor agonists at 76%). SSRIs and antidepressants performed near or below chance (50%), with cromolyn sodium the worst-performing treatment at 35% positive. Recommendations are tiered by sample size and statistical significance using Wilson score confidence intervals and binomial tests against a 50% baseline.

## 1. The Data: Who Are These Patients?

This analysis draws from r/covidlonghaulers, one of the most active Long COVID communities on Reddit. Before examining treatment outcomes, we need to understand what we are working with --- the data volume, time frame, and community composition shape everything that follows.

In [ ]:

# ── Data exploration ──
total_users = pd.read_sql("SELECT COUNT(DISTINCT user_id) FROM users", conn).iloc[0,0]
total_posts = pd.read_sql("SELECT COUNT(*) FROM posts", conn).iloc[0,0]
total_reports = pd.read_sql("SELECT COUNT(*) FROM treatment_reports", conn).iloc[0,0]
unique_reporters = pd.read_sql("SELECT COUNT(DISTINCT user_id) FROM treatment_reports", conn).iloc[0,0]
unique_drugs = pd.read_sql("SELECT COUNT(DISTINCT drug_id) FROM treatment_reports", conn).iloc[0,0]
date_range = pd.read_sql("SELECT date(min(post_date),'unixepoch') as d_min, date(max(post_date),'unixepoch') as d_max FROM posts", conn).iloc[0]

pct_reporters = round(100 * unique_reporters / total_users)

display(HTML(
    '<div style="background:#f8f9fa; padding:18px 24px; border-radius:8px; margin-bottom:12px;">'
    '<h3 style="margin-top:0;">Dataset Overview</h3>'
    '<table style="font-size:1.05em; border-collapse:collapse;">'
    f'<tr><td style="padding:4px 18px 4px 0;"><b>Data covers:</b></td><td>{date_range["d_min"]} to {date_range["d_max"]} (1 month)</td></tr>'
    f'<tr><td style="padding:4px 18px 4px 0;"><b>Community members:</b></td><td>{total_users:,}</td></tr>'
    f'<tr><td style="padding:4px 18px 4px 0;"><b>Posts analyzed:</b></td><td>{total_posts:,}</td></tr>'
    f'<tr><td style="padding:4px 18px 4px 0;"><b>Treatment reports extracted:</b></td><td>{total_reports:,}</td></tr>'
    f'<tr><td style="padding:4px 18px 4px 0;"><b>Users reporting treatments:</b></td><td>{unique_reporters:,} ({pct_reporters}% of community)</td></tr>'
    f'<tr><td style="padding:4px 18px 4px 0;"><b>Distinct treatments mentioned:</b></td><td>{unique_drugs:,}</td></tr>'
    '</table></div>'
))


In [ ]:

# ── Sentiment distribution pie chart ──
sent_counts = pd.read_sql(
    "SELECT sentiment, COUNT(*) as n FROM treatment_reports "
    "WHERE sentiment IN ('positive','negative','mixed','neutral') "
    "GROUP BY sentiment", conn)
sent_colors = {'positive': '#2ecc71', 'negative': '#e74c3c', 'mixed': '#f39c12', 'neutral': '#95a5a6'}
fig, ax = plt.subplots(figsize=(7, 7))
labels = sent_counts['sentiment'].values
sizes = sent_counts['n'].values
colors_list = [sent_colors.get(s, '#999') for s in labels]
wedges, texts, autotexts = ax.pie(
    sizes, labels=[f"{s.title()}\n(n={n:,})" for s, n in zip(labels, sizes)],
    colors=colors_list, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11}
)
for t in autotexts:
    t.set_fontsize(10)
    t.set_fontweight('bold')
ax.set_title("Overall Sentiment Distribution Across All Treatment Reports", fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


**What this means:** Two-thirds of all treatment reports are positive, reflecting a known reporting bias --- people who find something that works are more motivated to share. This 67% positive baseline is important context: a treatment needs to exceed this community-wide rate to stand out, and fall well below it to be considered underperforming. We use a conservative 50% baseline for binomial testing to avoid over-correcting for this bias.

## 2. The Rankings: Which Treatments Perform Best?

With the community baseline established, we can now rank treatments by their user-level positive outcome rate. Each user contributes exactly one data point per treatment (their average sentiment across all reports for that treatment), ensuring statistical independence. Treatments with fewer than 10 user-level reports are excluded as unreliable.

**Filtering applied:** Generic terms (e.g., "supplements," "medication") were removed because they are categories, not actionable treatments. Vaccines (COVID vaccine, Pfizer, Moderna, boosters) were excluded as causal-context contamination --- their negative sentiment reflects why users are in this community (perceived vaccine injury), not a treatment response. The generic term "antibiotics" was also excluded; specific antibiotics with sufficient data are retained. "Antihistamines" was excluded as a generic umbrella --- specific antihistamines (cetirizine, fexofenadine, famotidine) are analyzed individually.

In [ ]:

# ── User-level aggregation with filtering ──
CAUSAL_EXCLUSIONS = {
    'covid vaccine', 'flu vaccine', 'mmr vaccine', 'moderna vaccine',
    'mrna covid-19 vaccine', 'pfizer vaccine', 'vaccine', 'vaccine injection',
    'pfizer', 'booster', 'moderna'
}
GENERIC_EXCLUSIONS = GENERIC_TERMS | {'antibiotics', 'antihistamines'}
ALL_EXCLUSIONS = CAUSAL_EXCLUSIONS | GENERIC_EXCLUSIONS

placeholders = ','.join(['?' for _ in ALL_EXCLUSIONS])
user_drug = pd.read_sql(
    "SELECT tr.user_id, t.canonical_name as treatment, "
    "AVG(CASE tr.sentiment "
    "  WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5 "
    "  WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 "
    "  ELSE 0.0 END) as avg_score, "
    "COUNT(*) as n_reports, "
    "MAX(CASE WHEN tr.signal_strength = 'strong' THEN 1 ELSE 0 END) as has_strong "
    "FROM treatment_reports tr "
    "JOIN treatment t ON tr.drug_id = t.id "
    "GROUP BY tr.user_id, t.canonical_name",
    conn
)

# Apply exclusions
user_drug = user_drug[~user_drug['treatment'].isin(ALL_EXCLUSIONS)].copy()

# Classify outcome
user_drug['outcome'] = user_drug['avg_score'].apply(classify_outcome)

# Treatment-level summary
treat_summary = user_drug.groupby('treatment').agg(
    n_users=('user_id', 'nunique'),
    n_positive=('outcome', lambda x: (x == 'positive').sum()),
    n_negative=('outcome', lambda x: (x == 'negative').sum()),
    n_mixed=('outcome', lambda x: (x == 'mixed/neutral').sum()),
    mean_score=('avg_score', 'mean'),
).reset_index()

treat_summary['pos_rate'] = treat_summary['n_positive'] / treat_summary['n_users']
treat_summary['neg_rate'] = treat_summary['n_negative'] / treat_summary['n_users']

# Wilson CIs
treat_summary['ci_low'] = treat_summary.apply(lambda r: wilson_ci(int(r['n_positive']), int(r['n_users']))[0], axis=1)
treat_summary['ci_high'] = treat_summary.apply(lambda r: wilson_ci(int(r['n_positive']), int(r['n_users']))[1], axis=1)

# Binomial test vs 50% baseline
from scipy.stats import binomtest as _binomtest
def binom_p(row):
    result = _binomtest(int(row['n_positive']), int(row['n_users']), 0.5, alternative='greater')
    return result.pvalue

treat_summary['p_value'] = treat_summary.apply(binom_p, axis=1)

# Cohen's h effect size vs 50%
treat_summary['cohens_h'] = 2 * np.arcsin(np.sqrt(treat_summary['pos_rate'])) - 2 * np.arcsin(np.sqrt(0.5))

# NNT
treat_summary['nnt_val'] = treat_summary['pos_rate'].apply(lambda r: nnt(r, 0.5))

# Filter to n >= 10
treat_main = treat_summary[treat_summary['n_users'] >= 10].sort_values('pos_rate', ascending=False).reset_index(drop=True)


In [ ]:

# ── Forest plot: Top 30 treatments by positive rate with Wilson CIs ──
top30 = treat_main.head(30).sort_values('pos_rate', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 10))

y_pos = range(len(top30))
colors_fp = ['#2ecc71' if p < 0.05 else '#bdc3c7' for p in top30['p_value']]

ax.hlines(y_pos, top30['ci_low'], top30['ci_high'], colors='#95a5a6', linewidth=1.5)
ax.scatter(top30['pos_rate'], y_pos, c=colors_fp, s=80, zorder=5, edgecolors='white', linewidth=0.5)

ax.axvline(0.5, color='#e74c3c', linestyle='--', alpha=0.7, label='50% baseline (chance)')
ax.set_yticks(list(y_pos))
ax.set_yticklabels([f"{r['treatment']}  (n={int(r['n_users'])})" for _, r in top30.iterrows()], fontsize=9.5)
ax.set_xlabel("Positive Outcome Rate (Wilson 95% CI)", fontsize=11)
ax.set_title("Top 30 Treatments by Positive Outcome Rate", fontsize=14, fontweight='bold', pad=12)
ax.legend(
    handles=[
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=9, label='Statistically significant (p<0.05)'),
        plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='#bdc3c7', markersize=9, label='Not significant'),
        plt.Line2D([0],[0], color='#e74c3c', linestyle='--', label='50% baseline'),
    ],
    loc='lower right', fontsize=9, framealpha=0.9
)
ax.set_xlim(0.3, 1.05)
plt.tight_layout()
plt.show()


**What this means:** The forest plot shows each treatment's positive outcome rate as a dot, with horizontal lines indicating the 95% confidence interval. Green dots indicate treatments that are statistically significantly better than chance (50%); grey dots are not significant at p<0.05. Wider intervals reflect smaller sample sizes and less certainty. The top performers --- quercetin, magnesium, turmeric, electrolytes --- cluster above 85% positive, but some have small samples (n=10-13). Low dose naltrexone (LDN) is notable for combining a high positive rate (74%) with the largest sample size (n=183), giving it the narrowest confidence interval and highest statistical confidence.

## 3. Testing the Hypothesis: Do Top Treatments Beat Chance?

Visual rankings are a starting point, but overlapping confidence intervals can be misleading. This section applies formal statistical tests to determine which treatments have outcomes significantly better than the 50% baseline (binomial test), and calculates the Number Needed to Treat (NNT) --- how many patients need to try a treatment for one additional person to report benefit beyond chance.

In [ ]:

# ── Binomial test results table: treatments with n >= 20 ──
tested = treat_main[treat_main['n_users'] >= 20].copy()
tested = tested.sort_values('pos_rate', ascending=False).reset_index(drop=True)

# Display formatted table
display_df = tested[['treatment', 'n_users', 'n_positive', 'pos_rate', 'ci_low', 'ci_high', 'p_value', 'cohens_h', 'nnt_val']].copy()
display_df.columns = ['Treatment', 'Users', 'Positive', 'Pos Rate', 'CI Low', 'CI High', 'p-value', "Cohen's h", 'NNT']
display_df['Pos Rate'] = display_df['Pos Rate'].map('{:.1%}'.format)
display_df['CI Low'] = display_df['CI Low'].map('{:.1%}'.format)
display_df['CI High'] = display_df['CI High'].map('{:.1%}'.format)
display_df['p-value'] = display_df['p-value'].map(lambda x: '<0.001' if x < 0.001 else f'{x:.3f}')
display_df["Cohen's h"] = display_df["Cohen's h"].map('{:.2f}'.format)
display_df['NNT'] = display_df['NNT'].map(lambda x: f'{x:.1f}' if x is not None else '---')

styled = display_df.head(25).style.set_caption(
    "Treatment outcomes vs. 50% baseline (binomial test, one-sided) | Treatments with n >= 20 users"
).set_table_styles([
    {'selector': 'caption', 'props': [('font-size', '12px'), ('font-weight', 'bold'), ('padding', '8px')]},
    {'selector': 'th', 'props': [('font-size', '10px'), ('text-align', 'center')]},
    {'selector': 'td', 'props': [('font-size', '10px'), ('text-align', 'center')]},
])
display(styled)


**How to read this table:** The positive rate is the share of users whose average sentiment for a treatment was positive (score > 0.25). Cohen's h measures effect size --- values above 0.5 are considered medium effects, above 0.8 are large. NNT (Number Needed to Treat) tells you how many people would need to try the treatment for one extra person to report benefit beyond the 50% baseline. Lower NNT is better: an NNT of 2.3 for magnesium means roughly every 2-3 people who try it, one additional person reports positive outcomes beyond what we would expect by chance.

Magnesium, vitamin D, electrolytes, and quercetin all show large effect sizes (Cohen's h > 0.6) with highly significant p-values. Low dose naltrexone, with its massive sample (n=183), achieves significance despite a more modest positive rate --- its 74% still exceeds the baseline, and the large sample makes this conclusion robust.

In [ ]:

# ── Diverging bar chart: sentiment breakdown for top 25 by user count ──
top_by_n = treat_main.sort_values('n_users', ascending=False).head(25).sort_values('n_users', ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13, 9))
y = range(len(top_by_n))

pos_pct = top_by_n['n_positive'] / top_by_n['n_users']
neg_pct = top_by_n['n_negative'] / top_by_n['n_users']
mix_pct = top_by_n['n_mixed'] / top_by_n['n_users']

# Wilson CIs for positive side error bars
pos_ci_low = top_by_n['ci_low'].values
pos_ci_high = top_by_n['ci_high'].values
pos_err_low = pos_pct.values - pos_ci_low
pos_err_high = pos_ci_high - pos_pct.values

# Stacking: mixed first (innermost left), then negative (outermost left)
ax.barh(list(y), -mix_pct, left=0, color='#f39c12', height=0.65, label='Mixed/Neutral')
ax.barh(list(y), -neg_pct, left=-mix_pct, color='#e74c3c', height=0.65, label='Negative')
ax.barh(list(y), pos_pct, left=0, color='#2ecc71', height=0.65, label='Positive',
        xerr=[pos_err_low, pos_err_high], error_kw=dict(elinewidth=1, capsize=2, color='#666'))

ax.axvline(0, color='black', linewidth=0.8)
ax.set_yticks(list(y))
ax.set_yticklabels([f"{r['treatment']}  (n={int(r['n_users'])})" for _, r in top_by_n.iterrows()], fontsize=9.5)
ax.set_xlabel("Share of Users", fontsize=11)
ax.set_title("Sentiment Breakdown: 25 Most-Reported Treatments", fontsize=13, fontweight='bold', pad=12)

# Format x-axis as percentages
from matplotlib.ticker import FuncFormatter
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{abs(x):.0%}'))

ax.legend(loc='upper left', bbox_to_anchor=(0.0, -0.06), ncol=3, fontsize=10, framealpha=0.9)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


**What this means:** This chart shows the full sentiment breakdown for the 25 treatments with the most user reports. Bars extend right for positive outcomes and left for negative/mixed. Error bars on the positive side show 95% Wilson confidence intervals. Several patterns emerge: LDN dominates in sample size but shows a meaningful negative tail (20% of users). SSRIs are nearly split down the middle. Magnesium and electrolytes have unusually small negative tails, suggesting broad tolerability.

## 4. Treatment Categories: Patterns Across Drug Classes

Individual treatment rankings can miss the forest for the trees. Here we group treatments into functional categories to see whether entire classes outperform or underperform. This aggregation smooths out noise from individual small-sample treatments and reveals systematic patterns.

In [ ]:

# ── Categorize treatments ──
categories = {
    'Micronutrients & Minerals': ['magnesium', 'vitamin d', 'vitamin c', 'b12', 'b vitamins', 'b1',
                                   'electrolyte', 'potassium', 'iron', 'zinc', 'vitamin d3',
                                   'magnesium glycinate', 'coq10', 'alpha-lipoic acid', 'nad'],
    'Anti-inflammatory / Antioxidant': ['quercetin', 'turmeric', 'curcumin', 'omega-3', 'fish oil',
                                         'n-acetylcysteine', 'glutathione', 'taurine'],
    'Mast Cell / Antihistamine': ['ketotifen', 'famotidine', 'cetirizine', 'fexofenadine',
                                   'h1 antihistamine', 'h2 antihistamine', 'pepcid', 'loratadine',
                                   'cromolyn sodium', 'mast cell stabilizer', 'dao', 'benadryl',
                                   'mast cell activation syndrome'],
    'Immune Modulators': ['low dose naltrexone', 'nicotine', 'maraviroc', 'antivirals',
                           'paxlovid', 'fluvoxamine', 'stellate ganglion block', 'sgb',
                           'palmitoylethanolamide', 'pea'],
    'Cardiovascular / Autonomic': ['beta blocker', 'propranolol', 'ivabradine', 'metoprolol',
                                    'guanfacine'],
    'Metabolic / GLP-1': ['glp-1 receptor agonist', 'tirzepatide', 'zepbound', 'metformin',
                           'creatine'],
    'Psychiatric Medications': ['ssri', 'antidepressants', 'escitalopram', 'sertraline',
                                 'mirtazapine', 'gabapentin', 'benzodiazepine',
                                 'sleep medication', 'ADHD medication', 'stimulants'],
    'Supplements & Herbals': ['nattokinase', 'probiotics', 'melatonin', 'cbd', 'cannabis', 'weed',
                               'thc', 'aspirin', 'ibuprofen', 'Advil', 'nsaids', 'caffeine',
                               'l-arginine', 'bpc 157', 'prebiotic', 'peptide'],
    'Procedures': ['acupuncture', 'red light therapy', 'vagus nerve stimulation',
                    'hyperbaric oxygen therapy', 'hbot'],
}

# Reverse map
treat_to_cat = {}
for cat, treats in categories.items():
    for t in treats:
        treat_to_cat[t] = cat

cat_data = treat_main.copy()
cat_data['category'] = cat_data['treatment'].map(treat_to_cat)
cat_data = cat_data.dropna(subset=['category'])

cat_summary = cat_data.groupby('category').apply(
    lambda g: pd.Series({
        'treatments': len(g),
        'total_users': g['n_users'].sum(),
        'weighted_pos_rate': np.average(g['pos_rate'], weights=g['n_users']),
        'median_pos_rate': g['pos_rate'].median(),
    })
).reset_index().sort_values('weighted_pos_rate', ascending=True)

# Grouped bar chart: weighted vs median positive rate
fig, ax = plt.subplots(figsize=(12, 7))
y = range(len(cat_summary))
bar_h = 0.35

bars1 = ax.barh([i + bar_h/2 for i in y], cat_summary['weighted_pos_rate'], bar_h,
                label='Weighted mean positive rate', color='#3498db', alpha=0.85)
bars2 = ax.barh([i - bar_h/2 for i in y], cat_summary['median_pos_rate'], bar_h,
                label='Median positive rate', color='#9b59b6', alpha=0.75)

ax.axvline(0.5, color='#e74c3c', linestyle='--', alpha=0.6, label='50% baseline')
ax.set_yticks(list(y))
ylabels = [f"{r['category']}\n({int(r['treatments'])} treatments, n={int(r['total_users'])})"
           for _, r in cat_summary.iterrows()]
ax.set_yticklabels(ylabels, fontsize=9)
ax.set_xlabel("Positive Outcome Rate", fontsize=11)
ax.set_title("Positive Outcome Rate by Treatment Category", fontsize=13, fontweight='bold', pad=12)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.set_xlim(0, 1.0)
plt.tight_layout()
plt.show()


**What this means:** Micronutrients/minerals and anti-inflammatory/antioxidant supplements lead in positive outcomes, with weighted positive rates above 80%. Cardiovascular/autonomic medications also perform well --- beta blockers and ivabradine are generally well-tolerated by this community. Psychiatric medications are the clear underperformer, with both weighted and median positive rates hovering near 50% (chance level). The gap between the weighted mean and median for mast cell/antihistamine treatments suggests a few high-volume treatments (like famotidine) pull the average down relative to the best performers in the category (fexofenadine at 79%).

## 5. Counterintuitive Findings Worth Investigating

The rankings above tell a coherent story --- supplements and immune modulators do well, psychiatric medications struggle. But several findings in this data are surprising enough to warrant closer examination.

In [ ]:

# ── Finding 1: SSRIs struggle, but fluvoxamine is no better ──
ssri_drugs = ['ssri', 'fluvoxamine', 'sertraline', 'escitalopram']
ssri_data = treat_main[treat_main['treatment'].isin(ssri_drugs)].copy()

fluv = treat_main[treat_main['treatment'] == 'fluvoxamine'].iloc[0]
other_ssris = treat_main[treat_main['treatment'].isin(['ssri', 'sertraline', 'escitalopram'])]
other_n = int(other_ssris['n_users'].sum())
other_pos = int(other_ssris['n_positive'].sum())

from scipy.stats import fisher_exact as _fisher
table = [[int(fluv['n_positive']), int(fluv['n_users'] - fluv['n_positive'])],
         [other_pos, other_n - other_pos]]
odds_r, p_fish = _fisher(table)

display(HTML(
    '<div style="background:#fff3cd; padding:16px 20px; border-left:4px solid #ffc107; border-radius:4px; margin-bottom:16px;">'
    '<h4 style="margin-top:0; color:#856404;">Finding 1: Fluvoxamine --- the community\'s preferred SSRI --- performs no better than other SSRIs</h4>'
    '<p>Fluvoxamine (brand name Luvox) gained attention early in the pandemic as a potential COVID treatment based on its sigma-1 receptor activity. '
    f'It has become the most-discussed SSRI in the Long COVID community (n={int(fluv["n_users"])} users), yet its positive rate '
    f'({fluv["pos_rate"]:.0%}) is comparable to the broader SSRI category ({other_pos}/{other_n} = {other_pos/other_n:.0%} positive). '
    f'Fisher\'s exact test: OR={odds_r:.2f}, p={p_fish:.3f}. There is no evidence that fluvoxamine outperforms other SSRIs '
    'in this community despite its outsized reputation.</p></div>'
))


In [ ]:

# ── Finding 2: Nicotine patches rival LDN ──
nic = treat_main[treat_main['treatment'] == 'nicotine'].iloc[0]
ldn = treat_main[treat_main['treatment'] == 'low dose naltrexone'].iloc[0]

table2 = [[int(nic['n_positive']), int(nic['n_users'] - nic['n_positive'])],
          [int(ldn['n_positive']), int(ldn['n_users'] - ldn['n_positive'])]]
or2, p2 = _fisher(table2)

display(HTML(
    '<div style="background:#fff3cd; padding:16px 20px; border-left:4px solid #ffc107; border-radius:4px; margin-bottom:16px;">'
    '<h4 style="margin-top:0; color:#856404;">Finding 2: Nicotine patches rival the community\'s most-recommended treatment</h4>'
    '<p>Nicotine --- typically an addictive substance with known health risks --- shows a 71% positive rate among 82 users, '
    f'statistically indistinguishable from Low Dose Naltrexone\'s 74% positive rate among 183 users '
    f'(Fisher\'s exact: OR={or2:.2f}, p={p2:.3f}). This is striking because LDN has years of community advocacy behind it '
    'and is the most-discussed treatment in r/covidlonghaulers, while nicotine patches are a recent and unconventional addition '
    'to the Long COVID toolkit. A clinician would not expect a nicotine-based intervention to perform comparably to an '
    'established immune modulator.</p></div>'
))


In [ ]:

# ── Finding 3: Cromolyn sodium is the worst mast cell treatment ──
crom = treat_main[treat_main['treatment'] == 'cromolyn sodium'].iloc[0]
keto = treat_main[treat_main['treatment'] == 'ketotifen'].iloc[0]

table3 = [[int(crom['n_positive']), int(crom['n_users'] - crom['n_positive'])],
          [int(keto['n_positive']), int(keto['n_users'] - keto['n_positive'])]]
or3, p3 = _fisher(table3)

p1_ch = crom['pos_rate']
p2_ch = keto['pos_rate']
ch_crom = 2 * np.arcsin(np.sqrt(p1_ch)) - 2 * np.arcsin(np.sqrt(p2_ch))

display(HTML(
    '<div style="background:#fff3cd; padding:16px 20px; border-left:4px solid #ffc107; border-radius:4px; margin-bottom:16px;">'
    '<h4 style="margin-top:0; color:#856404;">Finding 3: Cromolyn sodium is the worst-performing mast cell treatment despite being a first-line stabilizer</h4>'
    '<p>MCAS (Mast Cell Activation Syndrome) is one of the most common co-occurring conditions in this community (75 users). '
    f'Cromolyn sodium, a first-line mast cell stabilizer commonly prescribed for MCAS, has the lowest positive rate of any treatment '
    f'with n>=20 at just {crom["pos_rate"]:.0%} (n={int(crom["n_users"])}). Meanwhile, ketotifen --- another mast cell stabilizer --- '
    f'performs at {keto["pos_rate"]:.0%} (n={int(keto["n_users"])}). The difference is statistically significant '
    f'(Fisher\'s exact: OR={or3:.2f}, p={p3:.3f}, Cohen\'s h={ch_crom:.2f}). '
    'This contradicts clinical guidelines where cromolyn is typically tried first due to its favorable side-effect profile. '
    'The data suggests that for Long COVID-related MCAS, the community experiences better outcomes with ketotifen and '
    'antihistamines than with cromolyn.</p></div>'
))


## 6. What Patients Are Saying *(experimental --- under development)*

The numbers above describe rates and probabilities. Here, selected quotes from community posts illustrate the lived experience behind those statistics. These are drawn from the raw post text joined with treatment reports.

In [ ]:

# ── Pull quotes for key treatments ──
def get_quotes(treatment_name, n=3, min_len=60, max_len=400):
    rows = pd.read_sql(
        "SELECT substr(p.body_text, 1, 350) as quote, date(p.post_date, 'unixepoch') as post_date "
        "FROM posts p "
        "JOIN treatment_reports tr ON tr.post_id = p.post_id "
        "JOIN treatment t ON tr.drug_id = t.id "
        "WHERE t.canonical_name = ? "
        f"AND length(p.body_text) > {min_len} AND length(p.body_text) < {max_len} "
        "ORDER BY RANDOM() LIMIT ?",
        conn, params=(treatment_name, n)
    )
    return rows

ldn_q = get_quotes('low dose naltrexone', 2)
nic_q = get_quotes('nicotine', 2)
ssri_q = get_quotes('ssri', 1)
crom_q = get_quotes('cromolyn sodium', 1)

parts = ['<div style="background:#f0f4f8; padding:18px 22px; border-radius:8px;">']

def add_quotes(title, pct, n, df, border_color):
    parts.append(f'<h4>{title} --- {pct}% positive (n={n})</h4>')
    for _, row in df.iterrows():
        text = row['quote'].strip()
        if len(text) > 250:
            text = text[:247] + '...'
        parts.append(
            f'<blockquote style="border-left:3px solid {border_color}; padding-left:12px; margin:8px 0; '
            f'color:#333; font-style:italic;">'
            f'\"{text}\" '
            f'<span style="color:#888; font-size:0.9em;">--- [{row["post_date"]}]</span></blockquote>'
        )

add_quotes('Low Dose Naltrexone (LDN)', 74, 183, ldn_q, '#2ecc71')
add_quotes('Nicotine patches', 71, 82, nic_q, '#2ecc71')
add_quotes('SSRIs', 50, 50, ssri_q, '#f39c12')
add_quotes('Cromolyn sodium', 35, 23, crom_q, '#e74c3c')

parts.append('</div>')
display(HTML('\n'.join(parts)))


## 7. Tiered Recommendations

Based on the statistical analysis, treatments are classified into three evidence tiers. **Strong** recommendations have at least 30 user reports and a statistically significant positive result (p<0.05). **Moderate** recommendations have at least 15 users or p<0.10. **Preliminary** findings have fewer than 15 users and should be treated as early signals only.

In [ ]:

# ── Assign tiers ──
def assign_tier(row):
    if row['n_users'] >= 30 and row['p_value'] < 0.05:
        return 'Strong'
    elif row['n_users'] >= 15 or row['p_value'] < 0.10:
        return 'Moderate'
    else:
        return 'Preliminary'

treat_main['tier'] = treat_main.apply(assign_tier, axis=1)

def classify_direction(row):
    if row['pos_rate'] >= 0.60 and row['p_value'] < 0.10:
        return 'Recommended'
    elif row['pos_rate'] < 0.50:
        return 'Not Recommended'
    else:
        return 'Inconclusive'

treat_main['direction'] = treat_main.apply(classify_direction, axis=1)

# ── Sensitivity check ──
strong_signal = pd.read_sql(
    "SELECT tr.user_id, t.canonical_name as treatment, "
    "AVG(CASE tr.sentiment WHEN 'positive' THEN 1.0 WHEN 'mixed' THEN 0.5 "
    "WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 ELSE 0.0 END) as avg_score "
    "FROM treatment_reports tr "
    "JOIN treatment t ON tr.drug_id = t.id "
    "WHERE tr.signal_strength = 'strong' "
    "GROUP BY tr.user_id, t.canonical_name",
    conn
)
strong_signal = strong_signal[~strong_signal['treatment'].isin(ALL_EXCLUSIONS)]
strong_signal['outcome'] = strong_signal['avg_score'].apply(classify_outcome)

ss_summary = strong_signal.groupby('treatment').agg(
    n_users_ss=('user_id', 'nunique'),
    pos_rate_ss=('outcome', lambda x: (x == 'positive').mean())
).reset_index()
ss_summary = ss_summary[ss_summary['n_users_ss'] >= 10]

sensitivity = treat_main[['treatment', 'pos_rate', 'n_users']].merge(
    ss_summary, on='treatment', how='inner'
)
sensitivity['rate_diff'] = abs(sensitivity['pos_rate'] - sensitivity['pos_rate_ss'])
fragile = sensitivity[sensitivity['rate_diff'] > 0.15]
n_robust = len(sensitivity) - len(fragile)

fragile_msg = ''
if len(fragile) > 0:
    fragile_msg = f'{len(fragile)} treatment(s) show fragile results: {", ".join(fragile["treatment"].values)}.'
else:
    fragile_msg = 'No treatments showed fragile results --- the main conclusions are robust.'

display(HTML(
    '<div style="background:#e8f5e9; padding:14px 18px; border-radius:6px; margin-bottom:12px;">'
    f'<b>Sensitivity check:</b> When restricted to strong-signal reports only, {n_robust} of {len(sensitivity)} treatments '
    f'with sufficient data maintain their ranking direction (positive rate shifts by less than 15 percentage points). '
    f'{fragile_msg}</div>'
))


In [ ]:

# ── Recommendation charts: one per tier ──
for tier_name, color_title in [('Strong', '#27ae60'), ('Moderate', '#f39c12'), ('Preliminary', '#3498db')]:
    tier_df = treat_main[(treat_main['tier'] == tier_name) & (treat_main['pos_rate'] >= 0.40)].copy()
    tier_df = tier_df.sort_values('pos_rate', ascending=True).tail(20)

    if len(tier_df) == 0:
        continue

    fig_h = max(3, len(tier_df) * 0.4 + 1)
    fig, ax = plt.subplots(figsize=(10, fig_h))
    y = range(len(tier_df))

    bar_colors = ['#2ecc71' if d == 'Recommended' else '#e74c3c' if d == 'Not Recommended' else '#95a5a6'
                  for d in tier_df['direction']]

    err_low = tier_df['pos_rate'].values - tier_df['ci_low'].values
    err_high = tier_df['ci_high'].values - tier_df['pos_rate'].values

    ax.barh(list(y), tier_df['pos_rate'], color=bar_colors, height=0.6,
            xerr=[err_low, err_high], error_kw=dict(elinewidth=1, capsize=2, color='#555'))
    ax.axvline(0.5, color='#e74c3c', linestyle='--', alpha=0.5)

    ax.set_yticks(list(y))
    ax.set_yticklabels([f"{r['treatment']}  (n={int(r['n_users'])})" for _, r in tier_df.iterrows()], fontsize=9)
    ax.set_xlabel("Positive Outcome Rate", fontsize=10)
    ax.set_title(f"{tier_name} Evidence Tier", fontsize=12, fontweight='bold', color=color_title, pad=10)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.set_xlim(0, 1.1)

    ax.legend(
        handles=[
            plt.Rectangle((0,0),1,1, fc='#2ecc71', label='Recommended'),
            plt.Rectangle((0,0),1,1, fc='#e74c3c', label='Not Recommended'),
            plt.Rectangle((0,0),1,1, fc='#95a5a6', label='Inconclusive'),
        ],
        loc='lower right', fontsize=8, framealpha=0.9
    )
    plt.tight_layout()
    plt.show()


**How to read these charts:** Green bars indicate treatments the data supports recommending (positive rate above 60%, p<0.10). Red bars indicate treatments that underperform the 50% baseline. Grey bars are inconclusive. Error bars show 95% Wilson confidence intervals --- wider bars mean less certainty. The red dashed line marks the 50% baseline (chance).

## 8. Conclusion

The Long COVID treatment landscape, as reflected in one month of r/covidlonghaulers data (2,827 users, 6,815 treatment reports), shows a clear hierarchy. The most effective treatments fall into two categories: **micronutrients and supplements** (magnesium, electrolytes, vitamin D, quercetin, B vitamins) and **immune modulators** (low dose naltrexone, nicotine patches). These are not fringe recommendations --- they represent consistent, statistically significant positive outcomes across hundreds of user-level data points.

Low dose naltrexone stands out as the single most important treatment in this dataset. Not because it has the highest positive rate (74% is good but not the best), but because it combines a strong positive signal with the largest sample size of any treatment (n=183), giving us high confidence that the finding is real. For a patient asking where to start, LDN is the most evidence-backed option in this community's experience.

The most surprising finding is nicotine's performance. With a 71% positive rate across 82 users, nicotine patches perform statistically on par with LDN --- a result that would raise eyebrows in any clinical setting. This does not mean patients should start smoking; the community uses pharmaceutical nicotine patches at low doses. But it does suggest that nicotinic acetylcholine receptor modulation deserves serious clinical investigation for Long COVID.

On the negative side, psychiatric medications --- particularly SSRIs and antidepressants --- consistently underperform. Fluvoxamine, despite its early-pandemic reputation as a COVID treatment, shows no advantage over generic SSRIs in this community (46% vs 50% positive). Cromolyn sodium, a first-line mast cell stabilizer, is the worst-performing treatment in the dataset at 35% positive. These findings do not mean these medications are useless --- they may address specific symptoms that patients do not frame as "positive outcomes" --- but they do suggest that the community's lived experience with these drugs is significantly worse than with supplements and immune modulators. Patients and clinicians should weigh this community evidence alongside clinical guidelines when making treatment decisions.

## 9. Research Limitations

1. **Selection bias:** Users of r/covidlonghaulers are not representative of all Long COVID patients. They skew toward those who are online, English-speaking, treatment-seeking, and motivated enough to participate in online communities. Patients who recovered quickly or who never sought treatment are underrepresented.

2. **Reporting bias:** People who find effective treatments are more motivated to share positive experiences. The community-wide 67% positive rate likely overstates true treatment efficacy. We use a 50% baseline for testing to partially address this, but the true population baseline is unknown.

3. **Survivorship bias:** Users still active in the community may have different treatment histories than those who left --- either because they recovered (and stopped posting) or because they became too ill to participate. Both exits distort the treatment landscape we observe.

4. **Recall bias:** Treatment reports are based on self-reported retrospective accounts. Users may misremember timing, dosage, or outcomes. They may attribute improvement to whichever treatment they most recently started, even if recovery was spontaneous or due to another factor.

5. **Confounding:** Most Long COVID patients try multiple treatments simultaneously. A user reporting positive outcomes with magnesium may also be taking LDN, following a diet protocol, and exercising more. We aggregate to the user level and test individual treatments, but we cannot isolate causal effects.

6. **No control group:** There is no untreated comparison group. The 50% baseline is an arbitrary reference point. Some patients improve without treatment (natural recovery), and some treatments may have placebo effects. Without randomization, we cannot distinguish treatment effects from placebo or natural history.

7. **Sentiment vs. efficacy:** Our outcome measure is community-reported sentiment, not clinical efficacy. A treatment may receive positive sentiment because it reduces one symptom while worsening others, because it is well-tolerated even if ineffective, or because community enthusiasm creates a halo effect. Negative sentiment may reflect side effects rather than lack of efficacy.

8. **Temporal snapshot:** This analysis covers a single month (March 11 -- April 10, 2026). Treatment sentiment may shift over time as new evidence emerges, as community attitudes evolve, or as the patient population turns over. Findings should be validated against longer time periods and other data sources.

In [ ]:

display(HTML(
    '<div style="font-size: 1.2em; font-weight: bold; font-style: italic; '
    'padding: 20px; margin-top: 24px; text-align: center; '
    'background: #fff3e0; border-radius: 8px;">'
    'These findings reflect reporting patterns in online communities, '
    'not population-level treatment effects. This is not medical advice.'
    '</div>'
))
conn.close()
